# 📦 Export Trained Model — ExecuTorch for Android/Kotlin

This notebook takes the best checkpoint produced by `main.ipynb` and exports it as an  
**ExecuTorch `.pte` file** that can be loaded directly in a Kotlin Android app via the  
ExecuTorch runtime AAR.

### Export pipeline overview

```
best.pt  (PyTorch checkpoint)
    │
    ├─► ONNX (.onnx)              ← universal fallback / debugging
    │
    └─► ExecuTorch (.pte)         ← primary Android/Kotlin target
            │
            ├─ XNNPACK backend    ← optimised CPU inference on device
            └─ metadata YAML      ← class names & input specs for Kotlin
```

**Requirements (already in requirements.txt):**
- `ultralytics >= 8.4.0`
- `executorch >= 0.5.0`
- `torch >= 2.9.0`

---

## 📦 Install Export Dependencies

In [3]:
# Ensure all required packages are installed
# !pip install -q --upgrade ultralytics executorch

import torch
import ultralytics

print(f"PyTorch    : {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

PyTorch    : 2.11.0
Ultralytics: 8.4.37


In [9]:
import executorch
print(f"ExecuTorch : {executorch}")

ExecuTorch : <module 'executorch' (namespace) from ['/Users/musahibrahimali/Dev/Python/crop_disease_detection/.venv/lib/python3.13/site-packages/executorch']>


## 📁 Export Configuration

In [ ]:
from pathlib import Path

# ─── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
RUNS_DIR     = PROJECT_ROOT / "runs"
EXP_NAME     = "crop_disease_yolo26"

# Output directory for all exported artefacts
MODELS_DIR   = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# ─── Locate best checkpoint ─────────────────────────────────────────────────────
# Primary location — produced by main.ipynb training
BEST_PT = RUNS_DIR / EXP_NAME / "weights" / "best.pt"

# If training hasn't run yet, fall back to latest run
if not BEST_PT.exists():
    candidates = sorted(RUNS_DIR.glob("*/weights/best.pt"), key=lambda p: p.stat().st_mtime)
    if candidates:
        BEST_PT = candidates[-1]
        print(f"⚠️  Using fallback checkpoint: {BEST_PT}")
    else:
        raise FileNotFoundError(
            "No trained model found. Please run main.ipynb first to train the model."
        )

print(f"Source checkpoint : {BEST_PT}")
print(f"Export directory  : {MODELS_DIR}")

# ─── Input size (must match training) ──────────────────────────────────────────
IMG_SIZE = 640

# ─── Class metadata ────────────────────────────────────────────────────────────
CLASS_NAMES = [
    'Corn_Cercospora_Leaf_Spot', 'Corn_Common_Rust', 'Corn_Healthy',
    'Corn_Northern_Leaf_Blight', 'Corn_Streak',
    'Pepper_Bacterial_Spot', 'Pepper_Cercospora', 'Pepper_Early_Blight',
    'Pepper_Fusarium', 'Pepper_Healthy', 'Pepper_Late_Blight',
    'Pepper_Leaf_Blight', 'Pepper_Leaf_Curl', 'Pepper_Leaf_Mosaic',
    'Pepper_Septoria',
    'Tomato_Bacterial_Spot', 'Tomato_Early_Blight', 'Tomato_Fusarium',
    'Tomato_Healthy', 'Tomato_Late_Blight', 'Tomato_Leaf_Curl',
    'Tomato_Mosaic', 'Tomato_Septoria'
]

# OOD rejection threshold — baked into the metadata YAML for the Kotlin app
CONF_THRESHOLD = 0.50

print(f"\n✅  Config ready — {len(CLASS_NAMES)} classes, input {IMG_SIZE}×{IMG_SIZE}")

## 🔬 Load & Verify Model

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO(str(BEST_PT))

# Quick sanity check — run inference on a dummy tensor
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
print("Model loaded successfully.")
print(f"  Task      : {model.task}")
print(f"  Classes   : {model.names}")
print(f"  Input size: {IMG_SIZE}×{IMG_SIZE}")

# Confirm class list matches our expected 23 classes
assert len(model.names) == len(CLASS_NAMES), (
    f"Class count mismatch: model has {len(model.names)}, expected {len(CLASS_NAMES)}"
)
print("\n✅  Model verification passed.")

## 📄 Export to ONNX (Universal Fallback)

ONNX is exported first as it serves two purposes:
1. A universal fallback that works on any platform
2. A useful intermediate format for debugging

In [ ]:
import shutil

print("Exporting to ONNX ...")
onnx_result = model.export(
    format   = "onnx",
    imgsz    = IMG_SIZE,
    dynamic  = False,      # static batch=1 for mobile inference
    simplify = True,       # graph simplification
    opset    = 17,
    half     = False,      # FP32 for broad hardware compat
)

# Move to models/ directory
src_onnx = Path(str(onnx_result))
dst_onnx = MODELS_DIR / "crop_disease_yolo26.onnx"
shutil.copy2(src_onnx, dst_onnx)

size_mb = dst_onnx.stat().st_size / 1_048_576
print(f"\n✅  ONNX export complete")
print(f"   └─ {dst_onnx}  ({size_mb:.1f} MB)")

## 🤖 Export to ExecuTorch (.pte) — Android/Kotlin Target

ExecuTorch is Meta's next-generation on-device inference runtime. The exported `.pte` file:
- Runs on Android via the **XNNPACK** backend (optimised for ARM CPUs)
- Can optionally target GPU via Vulkan backend
- Is loaded in Kotlin with `Module.load("crop_disease_yolo26.pte")`

> **Note:** Ultralytics `model.export(format="executorch")` produces a directory containing the `.pte` file plus a metadata YAML. Both are moved to `models/`.

In [ ]:
import shutil
from pathlib import Path

print("Exporting to ExecuTorch (.pte) — this may take several minutes ...")
print("The model will be optimised for XNNPACK (ARM CPU inference on Android)\n")

# ─── Export via Ultralytics ────────────────────────────────────────────────────
# Ultralytics handles the full pipeline:
#   best.pt → torch.export.export() → ExecuTorch lowering → .pte
et_result = model.export(
    format = "executorch",
    imgsz  = IMG_SIZE,
    half   = False,        # FP32 — use half=True if you want FP16 on supported devices
)

# Ultralytics saves the .pte inside a subfolder; locate it
et_path = Path(str(et_result))

# Find .pte file — it may be the result itself or inside a directory
if et_path.is_dir():
    pte_files = list(et_path.rglob("*.pte"))
    pte_src   = pte_files[0] if pte_files else None
else:
    pte_src = et_path if et_path.suffix == ".pte" else None

if pte_src and pte_src.exists():
    dst_pte = MODELS_DIR / "crop_disease_yolo26.pte"
    shutil.copy2(pte_src, dst_pte)
    size_mb = dst_pte.stat().st_size / 1_048_576
    print(f"\n✅  ExecuTorch export complete")
    print(f"   └─ {dst_pte}  ({size_mb:.1f} MB)")
else:
    print("⚠️  Could not locate .pte file automatically.")
    print(f"   Check the export directory: {et_path}")
    print("   Manually copy the .pte file to:", MODELS_DIR)

## 🗂️ Write Metadata YAML for Kotlin

The Kotlin app needs to know the class names, input size, and confidence threshold without hard-coding them. We write a companion YAML alongside the `.pte` file.

In [ ]:
import yaml
from datetime import datetime

metadata = {
    "model_name"       : "crop_disease_yolo26",
    "architecture"     : "YOLO26",
    "task"             : "object_detection",
    "exported_at"      : datetime.utcnow().isoformat() + "Z",
    "input_size"       : IMG_SIZE,
    "input_channels"   : 3,
    "num_classes"      : len(CLASS_NAMES),
    "class_names"      : CLASS_NAMES,
    "conf_threshold"   : CONF_THRESHOLD,
    "iou_threshold"    : 0.45,
    "crops_covered"    : ["Corn", "Pepper", "Tomato"],
    "notes": (
        "Trained on Ghana Crop Disease Challenge v2 (Roboflow). "
        "Hard-negative mining and label_smoothing=0.1 applied to reduce "
        "false positives on non-crop images. "
        "Apply conf_threshold at inference time before displaying results."
    ),
    "android_integration": {
        "runtime"         : "ExecuTorch",
        "backend"         : "XNNPACK",
        "pte_file"        : "crop_disease_yolo26.pte",
        "input_normalize" : {"mean": [0.0, 0.0, 0.0], "std": [255.0, 255.0, 255.0]},
        "input_format"    : "NCHW_RGB",
    }
}

meta_path = MODELS_DIR / "model_metadata.yaml"
with open(meta_path, "w") as f:
    yaml.dump(metadata, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"Metadata written → {meta_path}")
print("\nContents:")
print(yaml.dump(metadata, default_flow_style=False, sort_keys=False, allow_unicode=True))

## 📂 Models Folder Summary

In [ ]:
from pathlib import Path

print(f"📁  {MODELS_DIR}")
print("")
for f in sorted(MODELS_DIR.iterdir()):
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name:<45}  {size_mb:>8.2f} MB")

print("""
────────────────────────────────────────────────────────────
 File                              Purpose
────────────────────────────────────────────────────────────
 crop_disease_yolo26.pte          Android / Kotlin (primary)
 crop_disease_yolo26.onnx         Universal fallback / debug
 model_metadata.yaml              Class names + thresholds
────────────────────────────────────────────────────────────
""")

## 🤖 Kotlin Integration Guide

Add this section to your `README.md` or Android module documentation.

In [ ]:
guide = """
═══════════════════════════════════════════════════════════════════════════
 Android / Kotlin — ExecuTorch Integration Guide
═══════════════════════════════════════════════════════════════════════════

1. ADD DEPENDENCY (build.gradle.kts)
─────────────────────────────────────
   dependencies {
       implementation("org.pytorch:executorch-android:0.5.0")
   }

2. COPY MODEL TO ASSETS
────────────────────────
   Copy models/crop_disease_yolo26.pte and models/model_metadata.yaml
   into:  app/src/main/assets/

3. LOAD MODEL IN KOTLIN
────────────────────────
   import org.pytorch.executorch.EValue
   import org.pytorch.executorch.Module
   import org.pytorch.executorch.Tensor

   val module = Module.load(assetFilePath(context, "crop_disease_yolo26.pte"))

4. PRE-PROCESS IMAGE
─────────────────────
   // Resize bitmap to 640×640
   val resized = Bitmap.createScaledBitmap(bitmap, 640, 640, true)

   // Convert to float NCHW tensor, normalise [0, 255] → [0.0, 1.0]
   val inputTensor = TensorImageUtils.bitmapToFloat32Tensor(
       resized,
       floatArrayOf(0f, 0f, 0f),       // mean
       floatArrayOf(1f/255f, 1f/255f, 1f/255f)  // std
   )

5. RUN INFERENCE
─────────────────
   val outputs = module.forward(EValue.from(inputTensor))
   val outputTensor = outputs[0].toTensor()   // shape: [1, num_boxes, 4 + num_classes]
   val data = outputTensor.dataAsFloatArray

6. POST-PROCESS OUTPUT
───────────────────────
   The output is raw YOLO format: [x_center, y_center, width, height, class_scores...]
   You must:
     a) Filter boxes where max(class_scores) >= 0.50  (CONF_THRESHOLD)
     b) Scale bounding boxes from [0,1] back to pixel coordinates
     c) Apply Non-Maximum Suppression (IoU threshold = 0.45)
     d) Map class index → CLASS_NAMES from model_metadata.yaml

   Recommended library: https://github.com/ultralytics/ultralytics
   Post-processing reference: ultralytics.utils.ops.non_max_suppression

7. DISPLAY RESULTS
───────────────────
   Only show boxes where:
     - conf >= 0.50      (eliminates most OOD false positives)
     - class_name does not contain "Unknown"  (future-proofing)

═══════════════════════════════════════════════════════════════════════════
"""
print(guide)

# Also write a Kotlin snippet file for reference
kotlin_snippet = '''// CropDiseaseDetector.kt — ExecuTorch inference helper
// Drop this file into your Android project.

import android.content.Context
import android.graphics.Bitmap
import org.pytorch.executorch.EValue
import org.pytorch.executorch.Module
import org.pytorch.torchvision.TensorImageUtils

data class Detection(
    val classId: Int,
    val className: String,
    val confidence: Float,
    val x1: Float, val y1: Float,
    val x2: Float, val y2: Float
)

class CropDiseaseDetector(context: Context) {

    companion object {
        const val INPUT_SIZE     = 640
        const val CONF_THRESHOLD = 0.50f
        const val IOU_THRESHOLD  = 0.45f
        val CLASS_NAMES = listOf(
            "Corn_Cercospora_Leaf_Spot", "Corn_Common_Rust", "Corn_Healthy",
            "Corn_Northern_Leaf_Blight", "Corn_Streak",
            "Pepper_Bacterial_Spot", "Pepper_Cercospora", "Pepper_Early_Blight",
            "Pepper_Fusarium", "Pepper_Healthy", "Pepper_Late_Blight",
            "Pepper_Leaf_Blight", "Pepper_Leaf_Curl", "Pepper_Leaf_Mosaic",
            "Pepper_Septoria",
            "Tomato_Bacterial_Spot", "Tomato_Early_Blight", "Tomato_Fusarium",
            "Tomato_Healthy", "Tomato_Late_Blight", "Tomato_Leaf_Curl",
            "Tomato_Mosaic", "Tomato_Septoria"
        )
    }

    private val module: Module = Module.load(
        assetFilePath(context, "crop_disease_yolo26.pte")
    )

    fun detect(bitmap: Bitmap): List<Detection> {
        // 1. Pre-process
        val resized = Bitmap.createScaledBitmap(bitmap, INPUT_SIZE, INPUT_SIZE, true)
        val inputTensor = TensorImageUtils.bitmapToFloat32Tensor(
            resized,
            floatArrayOf(0f, 0f, 0f),
            floatArrayOf(1f / 255f, 1f / 255f, 1f / 255f)
        )

        // 2. Inference
        val output = module.forward(EValue.from(inputTensor))[0].toTensor()
        val data   = output.dataAsFloatArray
        val shape  = output.shape()   // [1, num_boxes, 4 + num_classes]
        val numBoxes    = shape[1].toInt()
        val numFields   = shape[2].toInt()
        val numClasses  = numFields - 4

        // 3. Decode + confidence filter
        val candidates = mutableListOf<Detection>()
        for (i in 0 until numBoxes) {
            val offset = i * numFields
            val cx = data[offset + 0]; val cy = data[offset + 1]
            val bw = data[offset + 2]; val bh = data[offset + 3]

            var maxConf = 0f; var classId = -1
            for (c in 0 until numClasses) {
                val s = data[offset + 4 + c]
                if (s > maxConf) { maxConf = s; classId = c }
            }

            if (maxConf < CONF_THRESHOLD) continue  // OOD guard

            // Convert cx,cy,bw,bh (normalised) → pixel x1,y1,x2,y2
            val w = bitmap.width.toFloat(); val h = bitmap.height.toFloat()
            candidates.add(Detection(
                classId, CLASS_NAMES[classId], maxConf,
                (cx - bw/2f) * w, (cy - bh/2f) * h,
                (cx + bw/2f) * w, (cy + bh/2f) * h
            ))
        }

        // 4. NMS (greedy)
        return applyNMS(candidates, IOU_THRESHOLD)
    }

    private fun applyNMS(dets: List<Detection>, iouThr: Float): List<Detection> {
        val sorted   = dets.sortedByDescending { it.confidence }
        val selected = mutableListOf<Detection>()
        val suppressed = BooleanArray(sorted.size)
        for (i in sorted.indices) {
            if (suppressed[i]) continue
            selected.add(sorted[i])
            for (j in i + 1 until sorted.size) {
                if (!suppressed[j] && iou(sorted[i], sorted[j]) > iouThr)
                    suppressed[j] = true
            }
        }
        return selected
    }

    private fun iou(a: Detection, b: Detection): Float {
        val ix1 = maxOf(a.x1, b.x1); val iy1 = maxOf(a.y1, b.y1)
        val ix2 = minOf(a.x2, b.x2); val iy2 = minOf(a.y2, b.y2)
        val inter = maxOf(0f, ix2 - ix1) * maxOf(0f, iy2 - iy1)
        val unionA = (a.x2 - a.x1) * (a.y2 - a.y1)
        val unionB = (b.x2 - b.x1) * (b.y2 - b.y1)
        return inter / (unionA + unionB - inter + 1e-6f)
    }

    private fun assetFilePath(context: Context, assetName: String): String {
        val file = java.io.File(context.filesDir, assetName)
        if (!file.exists()) {
            context.assets.open(assetName).use { inp ->
                file.outputStream().use { out -> inp.copyTo(out) }
            }
        }
        return file.absolutePath
    }
}
'''

kotlin_file = MODELS_DIR / "CropDiseaseDetector.kt"
kotlin_file.write_text(kotlin_snippet)
print(f"\nKotlin sample saved → {kotlin_file}")
print("Copy CropDiseaseDetector.kt into your Android app's Kotlin source set.")

## ✅ Verify Exported Files

Loads the ONNX model and runs a dummy forward pass to confirm the export is valid  
(ExecuTorch `.pte` verification requires the Android runtime).

In [ ]:
import numpy as np

# ─── Verify ONNX ───────────────────────────────────────────────────────────────
try:
    import onnxruntime as ort

    session = ort.InferenceSession(str(dst_onnx), providers=['CPUExecutionProvider'])
    dummy_input = np.zeros((1, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    input_name  = session.get_inputs()[0].name
    outputs     = session.run(None, {input_name: dummy_input})

    print("✅  ONNX verification passed")
    for i, o in enumerate(outputs):
        print(f"   Output[{i}] shape : {o.shape}")
except ImportError:
    print("ℹ️   onnxruntime not installed (pip install onnxruntime) — skipping ONNX verification")
except Exception as e:
    print(f"⚠️  ONNX verification failed: {e}")

# ─── Verify .pte exists ────────────────────────────────────────────────────────
pte_file = MODELS_DIR / "crop_disease_yolo26.pte"
if pte_file.exists():
    size_mb = pte_file.stat().st_size / 1_048_576
    print(f"\n✅  ExecuTorch .pte present: {pte_file.name}  ({size_mb:.1f} MB)")
    print("   → Copy to your Android app's assets/ folder to deploy.")
else:
    print("\n⚠️  crop_disease_yolo26.pte not found — check the export cell above.")

print("\n─── All exported artefacts ─────────────────────────────────────────────")
for f in sorted(MODELS_DIR.iterdir()):
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name:<50}  {size_mb:>8.2f} MB")

## 📋 Export Summary

| File | Purpose | Target |
|------|---------|--------|
| `models/crop_disease_yolo26.pte` | **Primary Android model** — ExecuTorch runtime | Kotlin / Android |
| `models/crop_disease_yolo26.onnx` | Universal ONNX fallback | Any ONNX runtime |
| `models/model_metadata.yaml` | Class names, thresholds, input spec | Kotlin / Android |
| `models/CropDiseaseDetector.kt` | Ready-to-use Kotlin inference helper | Android Studio |

### Android deployment checklist

- [ ] Copy `crop_disease_yolo26.pte` → `app/src/main/assets/`
- [ ] Copy `model_metadata.yaml` → `app/src/main/assets/`
- [ ] Copy `CropDiseaseDetector.kt` → your Kotlin source set
- [ ] Add ExecuTorch dependency: `org.pytorch:executorch-android:0.5.0`
- [ ] Add TorchVision dependency: `org.pytorch:pytorch_android_torchvision:2.9.0`
- [ ] Call `detector.detect(bitmap)` — returns a `List<Detection>`
- [ ] Only render detections where `detection.confidence >= 0.50` (already filtered)

---
> **OOD reminder:** The `CONF_THRESHOLD = 0.50` in `CropDiseaseDetector.kt` is your primary guard against false positives on non-crop images. Raise it to `0.65` if you still see spurious detections in your app.